# NBA 3.1 Combine Physician Data

Joins the 3.1 feature tables listed in `NBA_3_1_Combine_Input_Tables_Plan_v1.md` onto one row per HCP x claims cohort (`HCP_COHORT_ID = PROVIDER_ID_c<n>`, cohorts c1-c28), one `COHORT_SET` at a time. It then adds calendar features, imputes nulls, and writes the combined Delta table (features saved as float) and a data dictionary to the 3.1 volume only.

**Sources (49 tables, 17,381 features):** claims, patient events, Rx market, static HCP, field and non-personal promotion, FTP and Copay (market access), and Veeva Link to People (T1 / T2 activity and profile; T3 vendor scores and IDN / practice peers in the look-ahead-free `asof` version). Veeva folders also hold old offset7 months, so only c1-c28 are read; Veeva coverage status and expert type are turned into 0 / 1 columns.

**Sample list (optional):** set `SAMPLE_LIST` to a Delta table with one row per HCP x cohort (`PROVIDER_ID`, `HCP_COHORT_ID`, `COHORT_ID`, `SPLIT`, `SAMPLE_GROUP`, `IS_EXISTING_WRITER`, `WEIGHT_GROUP`, `WEIGHT_MONTH`, `WEIGHT`). Only the listed rows are combined, the six sample columns are carried into the output, and cohort sets with no listed rows are skipped. Cannot be combined with `TEST_HCP_SAMPLE`.

**Run order:** Setup -> Config -> Skeleton -> Input checks -> Imputation statistics -> Combine and write (loop over COHORT_SETs) -> Validation.
**Widgets:** `STEP` (see below), `RUN_NAME` (output run folder), `COHORT_SETS` (comma list or `all`), `TEST_HCP_SAMPLE` (0 = full universe), `SAMPLE_LIST` (empty = no sample list), `WRITE_OUTPUT` (`true` / `false`).

**STEP setting:** lets one notebook do each part of a parallel run.

| STEP | What it does |
| --- | --- |
| `all` (default) | Everything in one run, as before: input checks, data dictionary, every set in `COHORT_SETS` one after another; with `WRITE_OUTPUT = true` also a fresh output table and the validation (with `false` the validation cell stops by design) |
| `prepare` | Input checks, null `PROVIDER_ID` side file, data dictionary, and a fresh empty output table; then stops |
| `set` | Combines and writes only the sets in `COHORT_SETS` (replaces just those sets in the table); then stops |
| `final` | Validation of the written table: every set present with the expected rows and no duplicate keys, columns = dictionary, no nulls, one HCP traced to 7 sources |

`prepare`, `set` and `final` need `WRITE_OUTPUT = true`. Each `set` task replaces a different set, so several can run at the same time.

**Parallel job:** `Kerendia NBA 3.1 - Combine (full, parallel month groups)` (job id 676440226088217, manual only) runs `prepare`, then the 7 sets side by side (`set`, one task per set), then `final`. If one set fails, repair only that task. Job parameters: `RUN_NAME` (default `combined_2026_11_26_full`), `WRITE_OUTPUT`, `TEST_HCP_SAMPLE`, `SAMPLE_LIST`.

In [0]:
from pyspark.sql import functions as F
import builtins, re

dbutils.widgets.text("RUN_NAME", "combined_2026_11_26_full")
dbutils.widgets.text("COHORT_SETS", "all")
dbutils.widgets.text("TEST_HCP_SAMPLE", "0")
dbutils.widgets.dropdown("WRITE_OUTPUT", "false", ["true", "false"])
dbutils.widgets.text("SAMPLE_LIST", "")   # optional: Delta table of HCP x cohort rows to combine (empty = all rows)
# STEP (for the parallel job): prepare = input checks, dictionary, empty output table; set = combine + write the COHORT_SETS given;
# final = checks on the written table; all = everything in one run (as before)
dbutils.widgets.dropdown("STEP", "all", ["all", "prepare", "set", "final"])

RUN_NAME        = dbutils.widgets.get("RUN_NAME").strip()
COHORT_SETS_ARG = dbutils.widgets.get("COHORT_SETS").strip()
TEST_HCP_SAMPLE = int(dbutils.widgets.get("TEST_HCP_SAMPLE") or 0)
WRITE_OUTPUT    = dbutils.widgets.get("WRITE_OUTPUT") == "true"
SAMPLE_LIST     = dbutils.widgets.get("SAMPLE_LIST").strip().rstrip("/")
STEP            = dbutils.widgets.get("STEP")
assert WRITE_OUTPUT or STEP == "all", "STEP prepare / set / final need WRITE_OUTPUT = true"
FEATURE_TYPE    = "float"   # features saved as float (agreed 2026-11-26): ~25% smaller, ~7 significant digits
assert RUN_NAME, "RUN_NAME is required"
assert not (SAMPLE_LIST and TEST_HCP_SAMPLE), "use either SAMPLE_LIST or TEST_HCP_SAMPLE, not both"

# ---- Roots (3.1 volume only) ----
VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/"
NO    = VOL31 + "notebook_output/"
DEV   = VOL31 + "hcp_level_model/dev_pipeline/notebook_output/"
CLAIMS_RUN = DEV + "full_2026_11_13/hcp_claims/"          # latest full claims run (plan Q3)
MA    = DEV + "market_access_nba_3_1/"                    # FTP and Copay
VL    = DEV + "veeva_link_to_people/"                     # Veeva Link to People (OUT of the Veeva notebooks)

UNIVERSE_PATH = VOL31 + "HCP_Universe_without_call_hcps"   # 294,727 HCPs, parquet
COHORTS_PATH  = VOL31 + "cohorts/claims/cohorts.parquet"   # c1-c28

# ---- Outputs ----
if SAMPLE_LIST:
    assert SAMPLE_LIST.startswith(VOL31.rstrip("/")), f"sample list must be in the 3.1 volume: {SAMPLE_LIST}"
SAMPLE_SUFFIX   = (f"_sample_{SAMPLE_LIST.split('/')[-1]}/" if SAMPLE_LIST else
                   f"_test_sample_{TEST_HCP_SAMPLE}/" if TEST_HCP_SAMPLE else "")
OUTPUT_DIR      = f"{DEV}{RUN_NAME}/hcp_combined/" + SAMPLE_SUFFIX
COMBINED_PATH   = OUTPUT_DIR + "kerendia_nba_3_1_hcp_selected_features_data"
DICTIONARY_PATH = OUTPUT_DIR + "kerendia_nba_3_1_hcp_feature_dictionary"
NULL_PID_DIR    = OUTPUT_DIR + "claims_null_provider_id/"
INPUT_REPORT    = OUTPUT_DIR + "input_check_report"

for _p in (COMBINED_PATH, DICTIONARY_PATH, NULL_PID_DIR, INPUT_REPORT):
    assert _p.startswith(VOL31) and "kerendia_nba_3_volume" not in _p, f"refusing to write outside the 3.1 volume: {_p}"

print(f"STEP={STEP}  RUN_NAME={RUN_NAME}  COHORT_SETS={COHORT_SETS_ARG}  TEST_HCP_SAMPLE={TEST_HCP_SAMPLE}  SAMPLE_LIST={SAMPLE_LIST or '-'}  WRITE_OUTPUT={WRITE_OUTPUT}")
print("Output:", COMBINED_PATH)

In [0]:
# One entry per table of NBA_3_1_Combine_Input_Tables_Plan_v1.md, section 3.
# cohort_col = name of the cohort column in the source; it is renamed to COHORT_ID.
PP  = "04_Promotional_Features/personal_promotional_features/"
NPP = "04_Promotional_Features/non_personal_promotional_features/"
HCPF = "03_Feature_Engineering/03_HCP_level_features/hcp_level_features/"

def _t(name, category, sub, notebook, path, fmt, cohort_col, drop_cols=(), prep=None):
    # prep: optional function applied to the raw table right after it is read (filters, encodings, key mapping)
    return dict(name=name, category=category, sub_category=sub, notebook=notebook,
                path=path, fmt=fmt, cohort_col=cohort_col, drop_cols=list(drop_cols), prep=prep)

SOURCE_TABLES = [
    # ---- Claims, HCP monthly (NB01-NB06) ----
    _t("provider_claim_rx_monthly_cohort", "Claims", "Rx claims", HCPF + "01 Rx Claims Files/01_claim_rx_monthly_cohort",
       CLAIMS_RUN + "provider_claim_rx_monthly_cohort", "delta", "COHORT_ID"),
    _t("provider_claim_lab_monthly_cohort", "Claims", "Lab claims", HCPF + "02 Lab Claims Files/02_claim_lab_monthly_cohort",
       CLAIMS_RUN + "provider_claim_lab_monthly_cohort", "delta", "COHORT_ID"),
    _t("provider_claim_diagnosis_monthly_cohort", "Claims", "Dx claims", HCPF + "03 Diagnosis Claims Files/03_claim_diagnosis_monthly_cohort",
       CLAIMS_RUN + "provider_claim_diagnosis_monthly_cohort", "delta", "COHORT_ID"),
    # NB04 repeats 5 Rx totals of NB01: keep them from NB01
    _t("final_claims_monthly_cohort", "Claims", "Monthly claims", HCPF + "04 Monthly Cohort Claims Files/04_claims_monthly_cohort",
       CLAIMS_RUN + "final_claims_monthly_cohort", "delta", "COHORT_ID",
       drop_cols=["RUNNING_TOTAL_RX_CLAIMS", "TOTAL_RX_CLAIMS_IN_LAST_1_MONTH", "TOTAL_RX_CLAIMS_IN_LAST_3_MONTH",
                  "TOTAL_RX_CLAIMS_IN_LAST_6_MONTH", "TOTAL_RX_CLAIMS_IN_LAST_12_MONTH"]),
    _t("claim_rx_nps_monthly_cohort", "Claims", "Rx NPS claims", HCPF + "05 Rx NPS Claims Files/05_claim_rx_nps_monthly_cohort",
       CLAIMS_RUN + "claim_rx_nps_monthly_cohort", "delta", "COHORT_ID"),
    _t("provider_claim_eligible_patient_cohort", "Claims", "Eligible patients", HCPF + "06 Eligible HCP Claims Files/06_claim_eligible_hcp",
       CLAIMS_RUN + "provider_claim_eligible_patient_cohort", "delta", "COHORT_ID"),
]

# ---- Patient events (15); v1 code writes to NO (plan Q1) ----
for fam, nb in [("rx", "RX"), ("dx", "DX"), ("px", "PX"), ("lab", "LAB")]:
    kinds = ["first_occurrence", "frequency", "recency"] + (["hospitalization"] if fam in ("dx", "px") else [])
    for k in kinds:
        SOURCE_TABLES.append(_t(f"{fam}_{k}_features", "Patient events", f"{fam.upper()} {k.replace('_', ' ')}",
                                f"03_patient_features/Final Notebooks/{nb}_Patient_Features",
                                f"{NO}patient_features/{fam}/{fam}_{k}_features", "delta", "COHORT"))
SOURCE_TABLES.append(_t("claims_addon_patient_features", "Patient events", "Add-on", "03_patient_features/Final Notebooks/01_Additional_Patient_Features",
                        NO + "patient_features/addon/claims_addon_patient_features", "delta", "COHORT"))

# ---- Rx market (6) ----
for n, nb in [("nbrx", "26_Features_NBRx_nba31"), ("nrx", "27_Features_NRx_nba31"), ("trx", "28_Features_TRx_nba31"),
              ("competitor_share", "31_Competitor_drugs_NBRx_share_nba31")]:
    SOURCE_TABLES.append(_t(f"{n}_features", "Rx market", n.upper() if n != "competitor_share" else "Competitor share",
                            "03_HCP_Level_Features/" + nb, f"{DEV}hcp_features_v7_cohort_aligned/{n}_features", "delta", "cohort_id"))
SOURCE_TABLES += [
    _t("kerendia_prior_writing", "Rx market", "Prior writing", "03_HCP_Level_Features/10_kerendia_prior_writing",
       DEV + "hcp_features/kerendia_prior_writing", "delta", "cohort_id"),
    _t("adoption_features", "Rx market", "Adoption", "03_HCP_Level_Features/11_adoption_features",
       DEV + "hcp_features/adoption_features", "delta", "cohort_id"),
]

# ---- Static HCP (6) ----
SOURCE_TABLES += [
    _t("physician_attributes_features", "Static HCP", "Physician attributes", "06_Physician_Static_Features/07_physician_attributes_features",
       DEV + "hcp_features/physician_attributes_features", "delta", "cohort_id"),
    _t("census_features", "Static HCP", "Census / SDoH", "07_Census_Sdoh_Features/08_census_features",
       DEV + "hcp_features/census_features", "delta", "cohort_id"),
    _t("colocation_idn_features", "Static HCP", "Colocation / IDN", "09_Colocation_Features/33_colocation_idn_features",
       DEV + "hcp_features/colocation_idn_features", "delta", "cohort_id"),
    _t("claim_comorbidity", "Static HCP", "Comorbidity", "03_HCP_Level_Features/09_claim_comorbidity",
       DEV + "hcp_features/claim_comorbidity", "delta", "cohort_id"),
    _t("competitor_nbrx_share", "Static HCP", "Competitor NBRx share", "03_HCP_Level_Features/31_Competitor_drugs_NBRX_share",
       DEV + "hcp_features/competitor_nbrx_share", "delta", "cohort_id"),
    _t("market_event_features", "Static HCP", "Market events", "03_HCP_Level_Features/36_market_event_features",
       DEV + "hcp_features/market_event_features", "delta", "cohort_id"),
]

# ---- Promotional, personal (field activity) ----
# field_core_account holds every feature of field_core and of field_account_call (checked 2026-11-24), so only it is joined.
FA = NO + "field_activity_nba_3_1/kerendia_nba_3_1_hcp_model_"
SOURCE_TABLES += [
    _t("field_core_account", "Promotional - personal", "F2F calls + account calls", PP + "F2F/f2f_feature",
       FA + "field_core_account_features.parquet", "parquet", "COHORT"),
    _t("field_events", "Promotional - personal", "Speaker programs", PP + "Speaker Programs/speaker_program_feature",
       FA + "field_events_features.parquet", "parquet", "COHORT_ID"),
    _t("field_irep", "Promotional - personal", "iRep emails", PP + "Irep emails/irep_email_feature",
       FA + "field_irep_features.parquet", "parquet", "COHORT_ID"),
    _t("field_message", "Promotional - personal", "Messages", PP + "Message features/message_feature",
       FA + "field_message_features.parquet", "parquet", "COHORT_ID"),
    _t("field_msl", "Promotional - personal", "MSL", PP + "MSL/msl_feature",
       FA + "field_msl_features.parquet", "parquet", "COHORT_ID"),
    _t("field_sample", "Promotional - personal", "Samples", PP + "Samples/samples_feature",
       FA + "field_sample_features.parquet", "parquet", "COHORT_ID"),
]
EXCLUDED_TABLES = {   # kept for the duplicate check in section 4
    "field_core":         FA + "field_core_features.parquet",
    "field_account_call": FA + "field_account_call_features.parquet",
}

# ---- Promotional, non-personal ----
SOURCE_TABLES += [
    _t("hq_email", "Promotional - non-personal", "HQ email", NPP + "HQ Emails/15_promotional_features_HQ_Emails_new",
       NO + "hq_email_nba_3_1/kerendia_nba_3_1_hcp_model_hq_email_features.parquet", "parquet", "COHORT"),
    _t("digital_paid_media", "Promotional - non-personal", "Digital paid media", NPP + "Digital_Engagement/24_Features_Digital_NBA31",
       NO + "digital_engagement_nba_3_1/kerendia_nba_3_1_hcp_model_digital_paid_media_features.parquet", "parquet", "COHORT"),
    _t("high_intent", "Promotional - non-personal", "High intent signals", NPP + "High_Intent_Signal_Features/25_Features_High_Intent_Signals_NBA31",
       NO + "high_intent_signals_nba_3_1/kerendia_nba_3_1_hcp_model_high_intent_features.parquet", "parquet", "COHORT"),
    _t("dtc", "Promotional - non-personal", "DTC", "writer not found (plan Q2)",
       NO + "dtc_nba_3_1/kerendia_nba_3_1_hcp_model_dtc_features.parquet", "parquet", "COHORT"),
    _t("oe", "Promotional - non-personal", "OE", NPP + "OE/35_OE_Promotional_Features_NBA31",
       NO + "oe_nba_3_1/kerendia_nba_3_1_oe_promotional_features.parquet", "parquet", "COHORT"),
]

# ---- Market access: FTP and Copay (Delta, one row per HCP x cohort, cohort column COHORT) ----
FTPC = "04_Promotional_Features/FTP_and_Copay/"
SOURCE_TABLES += [
    _t("ftp", "Market access", "FTP", FTPC + "ftp_feature",
       MA + "kerendia_nba_3_1_hcp_model_ftp_features", "delta", "COHORT"),
    _t("copay", "Market access", "Copay", FTPC + "copay_feature",
       MA + "kerendia_nba_3_1_hcp_model_copay_features", "delta", "COHORT"),
]

# ---- Veeva Link to People (parquet, partitioned by COHORT_ID) ----
# The folders also hold 43 old offset7 cohorts: keep only c1-c28. T3 and cross use the look-ahead-free 'asof' version
# (Veeva QC doc F7 / F10: 'snapshot' uses the 2025-03-06 load for c1-c11 and keeps stale rows).
VEEVA_NB = "Veeva Link To People Analysis/Implementation/"
VEEVA_T12 = VL + "features_phase1/t1_t2/"
VEEVA_COVERAGE = ["NOT_IN_VEEVA", "IN_VEEVA_UNSCORED", "SCORED"]
VEEVA_EXPERT_TYPES = ["MENTAL_HEALTH_THERAPIST", "NON_PHYSICIAN_RESEARCHER", "NURSE", "NURSE_PRACTITIONER", "OPTOMETRIST",
                      "PHARMACIST", "PHYSICAL_THERAPIST", "PHYSICIAN", "PHYSICIAN_ASSISTANT", "SPEECH_THERAPIST",
                      "TECHNICIAN/TECHNOLOGIST"]   # values found 2026-11-26; null = not in Veeva
VEEVA_TEXT_VALUES = {"veeva_coverage_status": VEEVA_COVERAGE, "veeva_expert_type": VEEVA_EXPERT_TYPES}
_C_COHORT = F.col("COHORT_ID").rlike("^c[0-9]+$")

def _onehot(df, col, values, prefix):
    """One 0/1 column per text value (null text -> null -> filled later); the text column is dropped."""
    for v in values:
        df = df.withColumn(prefix + re.sub(r"[^A-Za-z0-9]+", "_", v).strip("_").lower(), (F.col(col) == v).cast("int"))
    return df.drop(col)

def _prep_veeva_t12(df):
    df = df.filter(_C_COHORT)
    df = _onehot(df, "veeva_coverage_status", VEEVA_COVERAGE, "veeva_cov_")
    return _onehot(df, "veeva_expert_type", VEEVA_EXPERT_TYPES, "veeva_expert_type_")

def _prep_veeva_cross(df):
    # cross is keyed by NPI only: PROVIDER_ID from t1_t2 (one NPI per provider and cohort, all rows mapped on 2026-11-26)
    npi_map = spark.read.parquet(VEEVA_T12).filter(_C_COHORT).select("NPI_NUMBER", "COHORT_ID", "PROVIDER_ID")
    return df.filter(_C_COHORT).join(npi_map, ["NPI_NUMBER", "COHORT_ID"], "inner")

VEEVA_META = ["NPI_NUMBER", "cohort_set", "veeva_window_end", "veeva_snapshot_id", "veeva_data_age_months", "t3_version"]
SOURCE_TABLES += [
    _t("veeva_t1_t2", "Veeva Link", "Activity and profile (T1 / T2)", VEEVA_NB + "20_phase1_t1_t2_features",
       VEEVA_T12, "parquet", "COHORT_ID", drop_cols=VEEVA_META, prep=_prep_veeva_t12),
    # veeva_coverage_status is the same as in t1_t2 on every row (checked 2026-11-26); veeva_decile_bin is a text band of veeva_best_decile
    _t("veeva_t3", "Veeva Link", "Vendor scores (T3, asof)", VEEVA_NB + "30_phase2_t3_features_and_tests",
       VL + "features_phase2/t3/version=asof/", "parquet", "COHORT_ID",
       drop_cols=VEEVA_META + ["veeva_coverage_status", "veeva_decile_bin", "veeva_score_load_date"],
       prep=lambda df: df.filter(_C_COHORT)),
    _t("veeva_cross", "Veeva Link", "IDN / practice expert peers (asof)", VEEVA_NB + "30_phase2_t3_features_and_tests",
       VL + "features_phase2/cross/version=asof/", "parquet", "COHORT_ID", drop_cols=VEEVA_META, prep=_prep_veeva_cross),
]

# Feature descriptions saved by the source notebooks (FEATURE, DESCRIPTION)
MANIFESTS = {
    "hq_email":           NO + "hq_email_nba_3_1/feature_manifest.parquet",
    "digital_paid_media": NO + "digital_engagement_nba_3_1/feature_manifest.parquet",
    "high_intent":        NO + "high_intent_signals_nba_3_1/feature_manifest.parquet",
    "dtc":                NO + "dtc_nba_3_1/feature_manifest.parquet",
}

# Key and metadata columns: dropped from every source, kept once from the skeleton
KEY_COLS = {"HCP_COHORT_ID", "PROVIDER_ID", "COHORT_ID", "COHORT"}
METADATA_COLS = {"COHORT_MONTH", "COHORT_SET", "CUTOFF_DATE", "DATASET", "OFFSET_DAYS", "PULL_DT",
                 "SVC_DT", "SVC_DT_M", "PREDICTION_MONTH", "LOOKBACK_START_DATE", "LOOKBACK_END_DATE",
                 "SAMPLE_CUTOFF_DATE", "ZIP5", "DMA_CODE", "STATE_NAME"}

# COHORT_SETs: same split as the claims tables
COHORT_SET_IDS = {
    "C1-C5_training":      [f"c{i}" for i in range(1, 6)],
    "C6-C10_training":     [f"c{i}" for i in range(6, 11)],
    "C11-C15_training":    [f"c{i}" for i in range(11, 16)],
    "C16-C20_training":    [f"c{i}" for i in range(16, 21)],
    "C21-C24_training":    [f"c{i}" for i in range(21, 25)],
    "C25-C26_validation":  ["c25", "c26"],
    "C27-C28_backtesting": ["c27", "c28"],
}
ALL_COHORTS = [c for ids in COHORT_SET_IDS.values() for c in ids]
TRAINING_COHORTS = [f"c{i}" for i in range(1, 25)]
assert sorted(ALL_COHORTS) == sorted(f"c{i}" for i in range(1, 29))
assert len({t["name"] for t in SOURCE_TABLES}) == len(SOURCE_TABLES)

RUN_SETS = list(COHORT_SET_IDS) if COHORT_SETS_ARG.lower() == "all" else [s.strip() for s in COHORT_SETS_ARG.split(",")]
assert all(s in COHORT_SET_IDS for s in RUN_SETS), f"unknown COHORT_SET in {RUN_SETS}"
print(f"{len(SOURCE_TABLES)} source tables; COHORT_SETs to run: {RUN_SETS}")

In [0]:
cohorts = (spark.read.parquet(COHORTS_PATH)
           .select(F.col("cohort_id").alias("COHORT_ID"), F.col("dataset").alias("DATASET"),
                   F.col("prediction_month").alias("PREDICTION_MONTH"), F.col("lookback_end_date").alias("LOOKBACK_END_DATE")))
set_map = spark.createDataFrame([(c, s) for s, ids in COHORT_SET_IDS.items() for c in ids], "COHORT_ID string, COHORT_SET string")
cohorts = cohorts.join(set_map, "COHORT_ID", "inner")
assert cohorts.count() == 28, "cohort file does not hold c1-c28"

# Placeholder HCP '0' is kept out of the main table (D13)
universe = (spark.read.parquet(UNIVERSE_PATH)
            .select(F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID")).distinct()
            .filter(F.col("PROVIDER_ID").isNotNull() & (F.col("PROVIDER_ID") != "0")))
if TEST_HCP_SAMPLE:
    universe = universe.orderBy(F.xxhash64("PROVIDER_ID")).limit(TEST_HCP_SAMPLE)

# Columns carried from the sample list into the output (identifiers and weights, not features)
SAMPLE_COLS = ["SPLIT", "SAMPLE_GROUP", "IS_EXISTING_WRITER", "WEIGHT_GROUP", "WEIGHT_MONTH", "WEIGHT"] if SAMPLE_LIST else []

if SAMPLE_LIST:
    # Only the listed HCP x cohort rows, with their split, sample group, writer flag and weights
    sample = spark.read.format("delta").load(SAMPLE_LIST)
    _missing = [c for c in ["PROVIDER_ID", "HCP_COHORT_ID", "COHORT_ID", *SAMPLE_COLS] if c not in sample.columns]
    assert not _missing, f"sample list is missing columns: {_missing}"
    sample = sample.select(F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID"), "HCP_COHORT_ID", "COHORT_ID", *SAMPLE_COLS)
    n_list = sample.count()
    _chk = sample.agg(F.countDistinct("HCP_COHORT_ID").alias("keys"),
                      F.sum((F.col("HCP_COHORT_ID") != F.concat_ws("_", "PROVIDER_ID", "COHORT_ID")).cast("int")).alias("bad_keys")).first()
    assert _chk["keys"] == n_list, "sample list has duplicate HCP_COHORT_ID"
    assert not _chk["bad_keys"], "sample list HCP_COHORT_ID is not PROVIDER_ID + '_' + COHORT_ID"
    skeleton = (sample.join(universe, "PROVIDER_ID", "inner")
                .join(F.broadcast(cohorts), "COHORT_ID", "inner")
                .select("HCP_COHORT_ID", "PROVIDER_ID", "COHORT_ID", "COHORT_SET", "DATASET", "PREDICTION_MONTH", "LOOKBACK_END_DATE", *SAMPLE_COLS))
    n_hcp = skeleton.select("PROVIDER_ID").distinct().count()
    n_skel = skeleton.count()
    print(f"Sample list rows: {n_list:,}   skeleton rows: {n_skel:,}   HCPs: {n_hcp:,}")
    assert n_skel == n_list, "some sample rows are not in the HCP universe or the cohort file"
    display(skeleton.groupBy("PROVIDER_ID", "SPLIT").count().groupBy("SPLIT")
            .agg(F.min("count").alias("min_cohorts_per_hcp"), F.max("count").alias("max_cohorts_per_hcp")))
else:
    skeleton = (universe.crossJoin(F.broadcast(cohorts))
                .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "COHORT_ID"))
                .select("HCP_COHORT_ID", "PROVIDER_ID", "COHORT_ID", "COHORT_SET", "DATASET", "PREDICTION_MONTH", "LOOKBACK_END_DATE"))
    n_hcp = universe.count()
    n_skel = skeleton.count()
    print(f"HCPs: {n_hcp:,}   skeleton rows: {n_skel:,}   expected: {n_hcp * 28:,}")
    assert n_skel == n_hcp * 28

# COHORT_SETs without skeleton rows (e.g. c27-c28 when they are not in the sample list) are skipped
_present = {r[0] for r in skeleton.select("COHORT_SET").distinct().collect()}
_skipped = [s for s in RUN_SETS if s not in _present]
RUN_SETS = [s for s in RUN_SETS if s in _present]
if _skipped:
    print(f"Skipped COHORT_SETs with no rows: {_skipped}")
print(f"COHORT_SETs to run: {RUN_SETS}")
display(skeleton.groupBy("COHORT_SET", "DATASET").count().orderBy("COHORT_SET"))

In [0]:
def read_source(t):
    """Raw source table with its cohort column renamed to COHORT_ID and HCP_COHORT_ID built if missing."""
    df = spark.read.format(t["fmt"]).load(t["path"])
    if t.get("prep"):
        df = t["prep"](df)
    if t["cohort_col"] != "COHORT_ID":
        df = df.withColumnRenamed(t["cohort_col"], "COHORT_ID")
    if "HCP_COHORT_ID" not in df.columns:      # claims tables
        # concat (not concat_ws): a null PROVIDER_ID gives a null key instead of a bare cohort id
        df = df.withColumn("HCP_COHORT_ID", F.concat(F.col("PROVIDER_ID").cast("string"), F.lit("_"), F.col("COHORT_ID")))
    return df

def feature_columns(t, columns):
    drop = {c.upper() for c in t["drop_cols"]} | METADATA_COLS | KEY_COLS
    return [c for c in columns if c.upper() not in drop]

def load_features(t, cohort_ids):
    """HCP_COHORT_ID + feature columns, for the given cohorts; null PROVIDER_ID and placeholder '0' removed."""
    df = read_source(t)
    df = df.filter(F.col("COHORT_ID").isin(cohort_ids) & F.col("PROVIDER_ID").isNotNull() & (F.col("PROVIDER_ID") != "0"))
    return df.select("HCP_COHORT_ID", *feature_columns(t, df.columns))

# Feature list per table (schema only) and the duplicate-name check: every feature must come from exactly one table
FEATURES = {}
for t in SOURCE_TABLES:
    FEATURES[t["name"]] = feature_columns(t, read_source(t).columns)
_owner = {}
_dups = []
for name, cols in FEATURES.items():
    for c in cols:
        if c.upper() in _owner:
            _dups.append((c, _owner[c.upper()], name))
        _owner[c.upper()] = name
assert not _dups, f"feature names in two tables: {_dups[:20]}"

# field_core_account must hold every feature of the two excluded field tables
_fca = set(FEATURES["field_core_account"])
for n, p in EXCLUDED_TABLES.items():
    _missing = [c for c in spark.read.parquet(p).columns if c.upper() not in METADATA_COLS | KEY_COLS and c not in _fca]
    assert not _missing, f"{n}: features not in field_core_account: {_missing}"

print(f"{sum(len(v) for v in FEATURES.values()):,} features from {len(FEATURES)} tables; no duplicate names")
display(spark.createDataFrame([(t["name"], t["category"], len(FEATURES[t["name"]])) for t in SOURCE_TABLES],
                              "table string, category string, n_features int"))

In [0]:
# Per table: rows, distinct keys, cohorts, null PROVIDER_ID, placeholder '0', and HCPs matched to the skeleton
# Runs once (STEP all / prepare); the month-group tasks rely on it.
if STEP not in ("all", "prepare"):
    print(f"STEP={STEP}: input checks skipped (done in the prepare task)")
else:
    skel_keys = skeleton.select("HCP_COHORT_ID")
    rows = []
    for t in SOURCE_TABLES:
        df = read_source(t)
        a = df.agg(F.count("*").alias("rows"),
                   F.countDistinct(F.when(F.col("PROVIDER_ID").isNotNull(), F.col("HCP_COHORT_ID"))).alias("keys"),
                   F.sort_array(F.collect_set("COHORT_ID")).alias("cohorts"),
                   F.sum(F.col("PROVIDER_ID").isNull().cast("int")).alias("null_provider_id"),
                   F.sum((F.col("PROVIDER_ID") == "0").cast("int")).alias("placeholder_0")).first()
        matched = df.select("HCP_COHORT_ID").join(skel_keys, "HCP_COHORT_ID", "inner").count()
        rows.append((t["name"], t["category"], a["rows"], a["keys"], len(a["cohorts"]),
                     sorted(a["cohorts"]) == sorted(ALL_COHORTS), int(a["null_provider_id"] or 0),
                     int(a["placeholder_0"] or 0), matched, builtins.round(matched / n_skel, 4)))

    input_report = spark.createDataFrame(rows, "table string, category string, rows long, distinct_keys long, n_cohorts int, "
                                               "cohorts_c1_c28 boolean, null_provider_id long, placeholder_0 long, "
                                               "matched_to_skeleton long, match_rate double")
    display(input_report)

    _bad = input_report.filter("NOT cohorts_c1_c28 OR rows - null_provider_id != distinct_keys").collect()
    assert not _bad, f"tables with wrong cohorts or duplicate keys: {[r['table'] for r in _bad]}"

    # Veeva text columns are one-hot encoded from a fixed list: a new value would be lost silently, so stop instead
    _raw_t12 = spark.read.parquet(VEEVA_T12).filter(_C_COHORT)
    for _c, _vals in VEEVA_TEXT_VALUES.items():
        _new = [r[0] for r in _raw_t12.select(_c).distinct().collect() if r[0] is not None and r[0] not in _vals]
        assert not _new, f"{_c}: values not in the one-hot list: {_new}"
    print("Veeva text values: all in the one-hot lists")

    if WRITE_OUTPUT:
        input_report.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(INPUT_REPORT)

In [0]:
# D13: rows that cannot join to the skeleton are saved to a side folder instead of being lost.
# The provider_ tables carry no NPI column, so the NPI_NUMBER mapping of D13 is still open (map from the NPI-level claims tables).
for t in ([t for t in SOURCE_TABLES if t["category"] == "Claims"] if STEP in ("all", "prepare") else []):
    nulls = read_source(t).filter(F.col("PROVIDER_ID").isNull())
    n = nulls.count()
    print(f"{t['name']}: {n:,} rows with null PROVIDER_ID")
    if n and WRITE_OUTPUT:
        nulls.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(NULL_PID_DIR + t["name"])
if STEP not in ("all", "prepare"):
    print(f"STEP={STEP}: null PROVIDER_ID side file skipped (done in the prepare task)")

In [0]:
# D14: 0 for counts / flags; -9999 for first occurrence; 9999 for days / months since and time to adopt.
# D10: medians from the training cohorts (c1-c24) only, applied to every set.
FIRST_OCC_FILL  = -9999.0
TIME_SINCE_FILL = 9999.0
RE_TIME_SINCE   = re.compile(r"(DAYS_SINCE|MONTHS_SINCE|TIME_TO_ADOPT|MIN_DAYS_TO_)", re.I)   # re.I: Veeva names are lower case
ONE_COLUMNS     = {"veeva_cov_not_in_veeva"}         # HCPs with no Veeva row are 'not in Veeva'
MEDIAN_TABLES   = {"census_features"}                 # SDoH: every column
MEDIAN_COLUMNS  = {"HCP_EXPERIENCE"}                  # physician attributes

def impute_rule(table, col):
    if col in ONE_COLUMNS:
        return "one"
    if col.startswith("MAX_EVENT_TIME_") and col.endswith("_FIRST_OCCURRENCE"):
        return "first_occurrence"
    if RE_TIME_SINCE.search(col):
        return "time_since"
    if table in MEDIAN_TABLES or col in MEDIAN_COLUMNS:
        return "median"
    return "zero"

RULES = {(name, c): impute_rule(name, c) for name, cols in FEATURES.items() for c in cols}

MEDIANS = {}
for name in {n for (n, _), r in RULES.items() if r == "median"}:
    t = next(t for t in SOURCE_TABLES if t["name"] == name)
    cols = [c for (n, c), r in RULES.items() if n == name and r == "median"]
    train = read_source(t).filter(F.col("COHORT_ID").isin(TRAINING_COHORTS) & F.col("PROVIDER_ID").isNotNull())
    row = train.agg(*[F.percentile_approx(F.col(c).cast("double"), 0.5).alias(c) for c in cols]).first()
    MEDIANS.update({c: (float(row[c]) if row[c] is not None else 0.0) for c in cols})

def fill_value(table, col):
    r = RULES[(table, col)]
    return {"first_occurrence": FIRST_OCC_FILL, "time_since": TIME_SINCE_FILL, "zero": 0.0, "one": 1.0}.get(r, MEDIANS.get(col, 0.0))

_cnt = {}
for r in RULES.values():
    _cnt[r] = _cnt.get(r, 0) + 1
print("Columns per imputation rule:", _cnt)
print(f"{len(MEDIANS)} medians from training cohorts")

In [0]:
# One row per feature: name, category, sub-category, definition, source notebook / table, imputation
descriptions = {}
for name, p in MANIFESTS.items():
    for r in spark.read.parquet(p).select("FEATURE", "DESCRIPTION").collect():
        descriptions[(name, r["FEATURE"])] = r["DESCRIPTION"]

DERIVED = [("CALENDAR_MONTH", "Month of the cohort lookback end date"),
           ("CALENDAR_YEAR", "Year of the cohort lookback end date"),
           ("MONTHS_SINCE_KER_LAUNCH", "Months from the Kerendia launch (2021-07-01) to the lookback end date")]

dict_rows = []
for t in SOURCE_TABLES:
    for c in FEATURES[t["name"]]:
        dict_rows.append((c, t["category"], t["sub_category"], descriptions.get((t["name"], c)),
                          t["notebook"], t["path"], RULES[(t["name"], c)], float(fill_value(t["name"], c))))
for c, d in DERIVED:
    dict_rows.append((c, "Derived in combine", "Calendar", d, "NBA_3_1_Combine_Physician_Data", None, "none", None))

feature_dictionary = spark.createDataFrame(dict_rows, "FEATURE_NAME string, CATEGORY string, SUB_CATEGORY string, DEFINITION string, "
                                                      "SOURCE_NOTEBOOK string, SOURCE_TABLE string, IMPUTE_RULE string, IMPUTE_VALUE double")
assert feature_dictionary.count() == feature_dictionary.select("FEATURE_NAME").distinct().count(), "feature in two rows"
assert feature_dictionary.filter("CATEGORY IS NULL").count() == 0
display(feature_dictionary.groupBy("CATEGORY").agg(F.count("*").alias("features"),
                                                   F.sum(F.col("DEFINITION").isNotNull().cast("int")).alias("with_definition")))
if WRITE_OUTPUT and STEP in ("all", "prepare"):   # written once; the month-group tasks do not touch it
    feature_dictionary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(DICTIONARY_PATH)

In [0]:
KER_LAUNCH = "2021-07-01"
ID_COLS = ["HCP_COHORT_ID", "PROVIDER_ID", "COHORT_ID", "COHORT_SET", "DATASET", "PREDICTION_MONTH", "LOOKBACK_END_DATE"] + SAMPLE_COLS
_clash = [c for cols in FEATURES.values() for c in cols if c.upper() in {s.upper() for s in SAMPLE_COLS}]
assert not _clash, f"source features with the same name as a sample column: {_clash}"

# Sample runs: keep only source rows whose key is in this set's skeleton before the join.
# Same result as the left join alone (rows not in the skeleton are dropped by it anyway), but far less data is joined.
SUBSET_RUN = bool(SAMPLE_LIST or TEST_HCP_SAMPLE)

def build_set(set_name):
    """Skeleton rows of one COHORT_SET left-joined with every source table, calendar features added, nulls imputed."""
    ids = COHORT_SET_IDS[set_name]
    out = skeleton.filter(F.col("COHORT_SET") == set_name)
    keys = F.broadcast(out.select("HCP_COHORT_ID"))
    for t in SOURCE_TABLES:
        feats = load_features(t, ids)
        if SUBSET_RUN:
            feats = feats.join(keys, "HCP_COHORT_ID", "left_semi")
        out = out.join(feats, "HCP_COHORT_ID", "left")

    # Impute in double, then save as FEATURE_TYPE (float)
    feature_exprs = [F.coalesce(F.col(c).cast("double"), F.lit(fill_value(t["name"], c))).cast(FEATURE_TYPE).alias(c)
                     for t in SOURCE_TABLES for c in FEATURES[t["name"]]]
    calendar = [F.month("LOOKBACK_END_DATE").cast(FEATURE_TYPE).alias("CALENDAR_MONTH"),
                F.year("LOOKBACK_END_DATE").cast(FEATURE_TYPE).alias("CALENDAR_YEAR"),
                F.floor(F.months_between("LOOKBACK_END_DATE", F.lit(KER_LAUNCH).cast("date"))).cast(FEATURE_TYPE).alias("MONTHS_SINCE_KER_LAUNCH")]
    return out.select(*ID_COLS, *calendar, *feature_exprs)

def create_table():
    """Empty combined table with the final schema, partitioned by COHORT_SET (fresh: replaces any earlier table at this path).
    Created once before the month-group tasks run side by side, so they never race to create it."""
    build_set(list(COHORT_SET_IDS)[0]).limit(0).write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").partitionBy("COHORT_SET").save(COMBINED_PATH)
    print("Empty combined table created:", COMBINED_PATH)

def write_set(df, set_name):
    """Overwrite only this COHORT_SET partition of the combined table.
    Parallel tasks each replace a different COHORT_SET partition, which Delta allows at the same time."""
    try:
        exists = spark.read.format("delta").load(COMBINED_PATH).limit(0) is not None
    except Exception:
        exists = False
    w = df.write.format("delta").mode("overwrite")
    if exists:
        w.option("replaceWhere", f"COHORT_SET = '{set_name}'").save(COMBINED_PATH)
    else:
        w.partitionBy("COHORT_SET").save(COMBINED_PATH)

In [0]:
# STEP prepare: create the empty output table and stop. STEP all: fresh table, then every set. STEP set: only COHORT_SETS.
if STEP in ("all", "prepare") and WRITE_OUTPUT:
    create_table()
if STEP == "prepare":
    dbutils.notebook.exit(f"prepared {COMBINED_PATH}")

set_report = []
for s in (RUN_SETS if STEP in ("all", "set") else []):
    df = build_set(s)
    expected = skeleton.filter(F.col("COHORT_SET") == s).count()
    if WRITE_OUTPUT:
        write_set(df, s)
        written = spark.read.format("delta").load(COMBINED_PATH).filter(F.col("COHORT_SET") == s)
        n, k = written.count(), written.select("HCP_COHORT_ID").distinct().count()
    else:
        n, k = df.count(), df.select("HCP_COHORT_ID").distinct().count()
    set_report.append((s, expected, n, k, len(df.columns)))
    print(f"{s}: expected {expected:,}  rows {n:,}  keys {k:,}  columns {len(df.columns):,}")
    assert n == expected == k, f"{s}: row gain / loss or duplicate keys after the joins"

if set_report:
    display(spark.createDataFrame(set_report, "COHORT_SET string, skeleton_rows long, rows long, distinct_keys long, columns int"))
if STEP == "set":
    dbutils.notebook.exit(f"written {RUN_SETS}")

In [0]:
# Runs on the written table (WRITE_OUTPUT = true): rows per set, nulls after imputation, one HCP traced back to its sources
assert WRITE_OUTPUT, "set WRITE_OUTPUT = true to validate the written table"
combined = spark.read.format("delta").load(COMBINED_PATH)

_per_set = combined.groupBy("COHORT_SET").agg(F.count("*").alias("rows"), F.countDistinct("HCP_COHORT_ID").alias("keys")).orderBy("COHORT_SET")
display(_per_set)
# Every COHORT_SET that was meant to run is written in full, once (the parallel tasks each write one set)
_got = {r["COHORT_SET"]: (r["rows"], r["keys"]) for r in _per_set.collect()}
_exp = {r["COHORT_SET"]: r["count"] for r in skeleton.groupBy("COHORT_SET").count().collect() if r["COHORT_SET"] in RUN_SETS}
_wrong = {s: (_exp[s], _got.get(s)) for s in _exp if _got.get(s) != (_exp[s], _exp[s])}
assert not _wrong, f"COHORT_SETs missing or with wrong rows / keys (expected, (rows, keys)): {_wrong}"
print(f"All {len(_exp)} COHORT_SETs written with the expected rows and no duplicate keys")

feat_cols = [c for c in combined.columns if c not in ID_COLS]
dict_names = {r[0] for r in spark.read.format("delta").load(DICTIONARY_PATH).select("FEATURE_NAME").collect()}
assert set(feat_cols) == dict_names, "combined columns and data dictionary differ"

# Nulls after imputation, on the first written set (one pass over all columns)
_s = RUN_SETS[0]
_nulls = (combined.filter(F.col("COHORT_SET") == _s)
          .select(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in feat_cols]).first().asDict())
_bad = {c: v for c, v in _nulls.items() if v}
print(f"{_s}: columns with nulls after imputation: {len(_bad)}")
assert not _bad, list(_bad)[:20]

# Trace one HCP x cohort through 3 families
_key = combined.filter(F.col("COHORT_SET") == _s).select("HCP_COHORT_ID").orderBy("HCP_COHORT_ID").first()[0]
for name in ["provider_claim_rx_monthly_cohort", "rx_frequency_features", "field_core_account", "ftp", "copay", "veeva_t1_t2", "veeva_cross"]:
    t = next(t for t in SOURCE_TABLES if t["name"] == name)
    cols = FEATURES[name][:3]
    src = read_source(t).filter(F.col("HCP_COHORT_ID") == _key).select(*cols).collect()
    got = combined.filter(F.col("HCP_COHORT_ID") == _key).select(*cols).first()
    for c in cols:
        s_val = float(src[0][c]) if src and src[0][c] is not None else fill_value(name, c)
        # features are saved as float: compare to ~7 significant digits
        assert abs(s_val - got[c]) <= 1e-6 * builtins.max(1.0, abs(s_val)), f"{_key} {name}.{c}: source {s_val} vs combined {got[c]}"
print(f"Traced {_key}: values match the source tables")